In [1]:
"""
Entry point: run the whole recommendation pipeline end-to-end.

Terminal:
    python run_pipeline.py                     # full run
    python run_pipeline.py --only train_models evaluate_models
    python run_pipeline.py --config configs/config.yaml --retries 2

Jupyter notebook (paste this whole file in a cell, or):
    from run_pipeline import main
    main()                                     # full run
    main(["--only", "evaluate_models"])        # subset

Schedule it (e.g. daily 02:00) with cron:
    0 2 * * * cd /path/to/recommendation-pipeline && .venv/bin/python run_pipeline.py >> logs/cron.log 2>&1
"""
import argparse
import os
import sys
from pathlib import Path


def _project_root() -> Path:
    """Folder that contains run_pipeline.py and src/ (works in scripts and notebooks)."""
    try:
        return Path(__file__).resolve().parent
    except NameError:                      # running inside Jupyter: no __file__
        here = Path.cwd()
        for cand in [here, *here.parents, here / "recommendation-pipeline"]:
            if (cand / "src").is_dir() and (cand / "configs").is_dir():
                return cand
        raise FileNotFoundError(
            "Could not find the project folder (the one containing src/ and configs/). "
            "Use os.chdir(r'...\\recommendation-pipeline') before running this cell.")


ROOT = _project_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.orchestration.pipeline import DAG, run_pipeline  # noqa: E402
from src.utils import load_config  # noqa: E402


def _in_notebook() -> bool:
    return "ipykernel" in sys.modules


def main(argv=None) -> int:
    ap = argparse.ArgumentParser(description=__doc__,
                                 formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--config", default=None, help="path to config YAML")
    ap.add_argument("--retries", type=int, default=1, help="retries per task")
    ap.add_argument("--only", nargs="*", choices=list(DAG), help="run a subset of tasks")

    if argv is None and _in_notebook():
        argv = []                          # ignore Jupyter's own --f=kernel.json argument
    args, _unknown = ap.parse_known_args(argv)

    record = run_pipeline(load_config(args.config), retries=args.retries, only=args.only)
    print(f"\nPipeline status: {record['status'].upper()}  "
          f"({record['duration_sec']}s)  ->  reports/runs/latest_run.png")
    return 0 if record["status"] == "success" else 1


if __name__ == "__main__":
    code = main()
    if not _in_notebook():                 # sys.exit would raise SystemExit inside Jupyter
        sys.exit(code)


2026-11-14 19:58:18,241 | INFO    | orchestrator | === Pipeline run 20261114T195818_2dbfd4 | 8 tasks: ingest_data -> validate_data -> prepare_data -> transform_data -> build_features -> build_serving_features -> train_models -> evaluate_models
2026-11-14 19:58:18,243 | INFO    | orchestrator | >>> ingest_data (attempt 1)
2026-11-14 19:58:19,865 | INFO    | ingestion.synthetic | Clean synthetic data: 1000 users, 600 items, 58715 interactions
2026-11-14 19:58:19,992 | INFO    | ingestion.synthetic | Wrote users (1010 rows) -> C:\Users\Dnyaneshwar\Downloads\recommendation-pipeline\recommendation-pipeline\data\raw\source\users.csv
2026-11-14 19:58:19,997 | INFO    | ingestion.synthetic | Wrote items (606 rows) -> C:\Users\Dnyaneshwar\Downloads\recommendation-pipeline\recommendation-pipeline\data\raw\source\items.csv
2026-11-14 19:58:20,184 | INFO    | ingestion.synthetic | Wrote interactions (59891 rows) -> C:\Users\Dnyaneshwar\Downloads\recommendation-pipeline\recommendation-pipeline\data

2026/11/14 19:58:30 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/14 19:58:30 INFO mlflow.store.db.utils: Updating database tables


2026-11-14 19:58:34,517 | INFO    | tracking | Experiment tracking backend: mlflow (sqlite:///mlflow.db)


2026/11/14 19:58:34 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet

2026/11/14 19:58:34 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably

2026-11-14 19:58:35,309 | INFO    | training | Trained popularity             in 0.1s (run b03402f82d984fb48226445df8b54241)
2026-11-14 19:58:40,917 | INFO    | training | Trained item_knn_cf            in 0.1s (run 279d7eaae3994ff6ab35fb8e2db92a14)
2026-11-14 19:59:19,734 | INFO    | training | Trained matrix_factorization   in 38.6s (run 1f13624a6af14bc8871337f4d6b2cb6c)
2026-11-14 19:59:21,210 | INFO    | training | Trained implicit_als           in 1.3s (run e9387c5600004dcf93df9fd746cc86b7)
2026-11-14 19:59:21,474 | INFO    | training | Trained hybrid_als_content     in 0.0s (run 8ff6d3b5a75646578902f6fef1768e67)
2026-11-14 19:59:21,499 | INFO    | orchestrator | <<< train_models succeeded in 55.6s
2026-11-14 19:59:21,501 | INFO    | orchestrator | >>> evaluate_models (attempt 1)
2026-11-14 19:59:25,819 | INFO    | tracking | Experiment tracking backend: mlflow (sqlite:///mlflow.db)
2026-11-14 19:59:26,070 | INFO    | evaluation | popularity             P@10=0.0730 R@10=0.1635 NDC